# Task description
- Classify the speakers of given features.
- Main goal: Learn how to use transformer.
- Baselines:
  - Easy: Run sample code and know how to use transformer.
  - Medium: Know how to adjust parameters of transformer.
  - Hard: Construct [conformer](https://arxiv.org/abs/2005.08100) which is a variety of transformer. 

- Other links
  - Kaggle: [link](https://www.kaggle.com/t/859c9ca9ede14fdea841be627c412322)
  - Slide: [link](https://speech.ee.ntu.edu.tw/~hylee/ml/ml2021-course-data/hw/HW04/HW04.pdf)
  - Data: [link](https://drive.google.com/file/d/1T0RPnu-Sg5eIPwQPfYysipfcz81MnsYe/view?usp=sharing)
  - Video (Chinese): [link](https://www.youtube.com/watch?v=EPerg2UnGaI)
  - Video (English): [link](https://www.youtube.com/watch?v=Gpz6AUvCak0)
  - Solution for downloading dataset fail.: [link](https://drive.google.com/drive/folders/13T0Pa_WGgQxNkqZk781qhc5T9-zfh19e?usp=sharing)

# Fix Random Seed

In [31]:
import numpy as np
import torch
import random

def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(87)

# Data

## Dataset
- Original dataset is [Voxceleb1](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/).
- The [license](https://creativecommons.org/licenses/by/4.0/) and [complete version](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/files/license.txt) of Voxceleb1.
- We randomly select 600 speakers from Voxceleb1.
- Then preprocess the raw waveforms into mel-spectrograms.

- Args:
  - data_dir: The path to the data directory.
  - metadata_path: The path to the metadata.
  - segment_len: The length of audio segment for training. 
- The architecture of data directory \\
  - data directory \\
  |---- metadata.json \\
  |---- testdata.json \\
  |---- mapping.json \\
  |---- uttr-{random string}.pt \\

- The information in metadata
  - "n_mels": The dimention of mel-spectrogram.
  - "speakers": A dictionary. 
    - Key: speaker ids.
    - value: "feature_path" and "mel_len"


For efficiency, we segment the mel-spectrograms into segments in the traing step.

In [32]:
import os
import json
import torch
import random
from pathlib import Path
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
 
 
class myDataset(Dataset):
  def __init__(self, data_dir, segment_len = 128):
    
    self.data_dir = data_dir
    self.segment_len = segment_len
 
    # Load the mapping from speaker neme to their corresponding id. id -> int
    mapping_path = Path(data_dir) / "mapping.json"
    mapping = json.load(mapping_path.open())
    self.speaker2id = mapping["speaker2id"]
 
    # Load metadata of training data. id : features
    metadata_path = Path(data_dir) / "metadata.json" 
    metadata = json.load(open(metadata_path))["speakers"]
 
    # Get the total number of speaker.
    self.speaker_num = len(metadata.keys())
    self.data = []
    for speaker in metadata.keys():
      for utterances in metadata[speaker]:
        self.data.append([utterances["feature_path"], self.speaker2id[speaker]])
 
  def __len__(self):
    return len(self.data)
 
  def __getitem__(self, index):
    
    feat_path, speaker = self.data[index]
    # Load preprocessed mel-spectrogram.
    mel = torch.load(os.path.join(self.data_dir, feat_path))
 
    # Segmemt mel-spectrogram into "segment_len" frames.
    if len(mel) > self.segment_len:
      # Randomly get the starting point of the segment.
      start = random.randint(0, len(mel) - self.segment_len)
      # Get a segment with "segment_len" frames.
      mel = torch.FloatTensor(mel[start:start+self.segment_len])
    else:
      mel = torch.FloatTensor(mel)
    # Turn the speaker id into long for computing loss later.
    
    speaker = torch.FloatTensor([speaker]).long()
    return mel, speaker
 
  def get_speaker_number(self):
    return self.speaker_num

## Dataloader
- Split dataset into training dataset(90%) and validation dataset(10%).
- Create dataloader to iterate the data.


In [33]:
import torch
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence


def collate_batch(batch):
  # Process features within a batch.
  """Collate a batch of data."""
  mel, speaker = zip(*batch)
  # Because we train the model batch by batch, we need to pad the features in the same batch to make their lengths the same.
  mel = pad_sequence(mel, batch_first=True, padding_value=-20)    # pad log 10^(-20) which is very small value.
  # mel: (batch size, length, 40)
  return mel, torch.FloatTensor(speaker).long()


def get_dataloader(data_dir, batch_size, n_workers):
  """Generate dataloader"""
  dataset = myDataset(data_dir)
  speaker_num = dataset.get_speaker_number()
  # Split dataset into training dataset and validation dataset
    
  trainlen = int(0.9 * len(dataset))
  # define a split list
  lengths = [trainlen, len(dataset) - trainlen]

  trainset, validset = random_split(dataset, lengths)

  train_loader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=collate_batch,
  )
    
  valid_loader = DataLoader(
    validset,
    batch_size=batch_size,
    num_workers=n_workers,
    drop_last=True,
    pin_memory=True,
    collate_fn=collate_batch,
  )

  return train_loader, valid_loader, speaker_num


# Model
- TransformerEncoderLayer:
  - Base transformer encoder layer in [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
  - Parameters:
    - d_model: the number of expected features of the input (required).

    - nhead: the number of heads of the multiheadattention models (required).

    - dim_feedforward: the dimension of the feedforward network model (default=2048).

    - dropout: the dropout value (default=0.1).

    - activation: the activation function of intermediate layer, relu or gelu (default=relu).

- TransformerEncoder:
  - TransformerEncoder is a stack of N transformer encoder layers
  - Parameters:
    - encoder_layer: an instance of the TransformerEncoderLayer() class (required).

    - num_layers: the number of sub-encoder-layers in the encoder (required).

    - norm: the layer normalization component (optional).

In [34]:
import torch
from torch import einsum
from torch import nn
from torch.nn import functional as F
import math

from einops import rearrange
from einops.layers.torch import Rearrange

class Swish(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return x * torch.sigmoid(x)

def exists(val):
    return val is not None

def default(val, d):
    return val if exists(val) else d

class Attention(nn.Module):
    def __init__(
        self,
        dim,
        heads = 8,
        dim_head = 64,
        dropout = 0.,
        max_pos_emb = 512
    ):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads= heads
        self.scale = dim_head ** -0.5
        self.to_q = nn.Linear(dim, inner_dim, bias = False)
        self.to_kv = nn.Linear(dim, inner_dim * 2, bias = False)
        self.to_out = nn.Linear(inner_dim, dim)

        self.max_pos_emb = max_pos_emb
        self.rel_pos_emb = nn.Embedding(2 * max_pos_emb + 1, dim_head)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        context = None,
        mask = None,
        context_mask = None
    ):
        n, device, h, max_pos_emb, has_context = x.shape[-2], x.device, self.heads, self.max_pos_emb, exists(context)
        context = default(context, x)

        q, k, v = (self.to_q(x), *self.to_kv(context).chunk(2, dim = -1)) # 拆分q,k,v向量
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = h), (q, k, v)) # 重新排列维度

        dots = einsum('b h i d, b h j d -> b h i j', q, k) * self.scale # 计算分数

        # shaw's relative positional embedding

        seq = torch.arange(n, device = device)
        dist = rearrange(seq, 'i -> i ()') - rearrange(seq, 'j -> () j')
        dist = dist.clamp(-max_pos_emb, max_pos_emb) + max_pos_emb
        rel_pos_emb = self.rel_pos_emb(dist).to(q)

        pos_attn = einsum('b h n d, n r d -> b h n r', q, rel_pos_emb) * self.scale

        dots = dots + pos_attn

        if exists(mask) or exists(context_mask):
            mask = default(mask, lambda: torch.ones(*x.shape[:2], device = device))
            context_mask = default(context_mask, mask) if not has_context else default(context_mask, lambda: torch.ones(*context.shape[:2], device = device))
            mask_value = -torch.finfo(dots.dtype).max
            mask = rearrange(mask, 'b i -> b () i ()') * rearrange(context_mask, 'b j -> b () () j')
            dots.masked_fill_(~mask, mask_value)

        attn = dots.softmax(dim = -1)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.to_out(out)
        return self.dropout(out)

    
class self_Attentive_pooling(nn.Module):
    def __init__(self, dim):
        super(self_Attentive_pooling, self).__init__()
        self.sap_linear = nn.Linear(dim, dim)
        self.attention = nn.Parameter(torch.FloatTensor(dim,1))
        torch.nn.init.normal_(self.attention, std=.02) # Normal Distribution 
        
    def forward(self, x):
        # x = x.permute(0, 2, 1)
        h = torch.tanh(self.sap_linear(x))
        w = torch.matmul(h, self.attention).squeeze(dim=2)   # [batch,d_length,d_model] * [d_model,1] = [batch,d_length,1] -> [batch,d_length]
        w = F.softmax(w, dim=1).view(x.size(0), x.size(1), 1) # [batch,d_length] -> [batch,d_length,1]
        x = torch.sum(x * w, dim=1) # 分配加权求和
        return x
    
 
class AMSoftmax(nn.Module): # Improved Softmax
    def __init__(self, in_feats, n_classes, m = 0.3, s = 15, annealing=False):
        super(AMSoftmax, self).__init__()
        self.linaer = nn.Linear(in_feats, n_classes, bias=False)
        self.m = m
        self.s = s
 
    def _am_logsumexp(self, logits):
        max_x = torch.max(logits, dim=-1)[0].unsqueeze(-1)
        term1 = (self.s*(logits - (max_x + self.m))).exp()
        term2 = (self.s * (logits - max_x)).exp().sum(-1).unsqueeze(-1) - (self.s*(logits-max_x)).exp()
        return self.s * max_x + (term2 + term1).log()
 
    def forward(self, *inputs):
 
        x_vector = F.normalize(inputs[0], p=2, dim=-1)
        self.linaer.weight.data = F.normalize(self.linaer.weight.data, p=2,dim=-1)
        logits = self.linaer(x_vector)
        scaled_logits = (logits-self.m)*self.s
        return scaled_logits - self._am_logsumexp(logits)
    
class Classifier(nn.Module): 
  def __init__(self, d_model = 224, n_spks = 600, dropout = 0.1):
    super().__init__()
    # Project the dimension of features from that of input into d_model.
    self.prenet = nn.Linear(40, d_model)

    # self.prenet = nn.Sequential(
    #     nn.Conv2d(1, d_model, kernel_size = 3, stride = 2),
    #     nn.ReLU(),
    #     nn.Conv2d(d_model, d_model, kernel_size = 3, stride = 2),
    #     nn.ReLU(),
    # )

    # self.project = nn.Linear(9*d_model, d_model)

    # TODO:
    self.FFM = nn.Sequential(
        nn.LayerNorm(d_model),
        nn.Linear(d_model, d_model),
        Swish(),
        nn.Dropout(dropout),
        nn.Linear(d_model, d_model),
        nn.Dropout(dropout)
    )
    
    self.LN = nn.LayerNorm(d_model)
    
    self.dropout = nn.Dropout(dropout)


    self.self_attention = Attention(dim = d_model, heads = 2, dim_head = 24, dropout = dropout)
    
    self.Conv = nn.Sequential(

        nn.Conv1d(d_model, d_model*2, kernel_size = 1), # Pointwise Conv1d
        nn.GLU(dim = 1),  # GLU activation //merge two tensors into one tensor [64,128,512]  -> [64,128,256]
        nn.Conv1d(d_model, d_model, kernel_size = 31, groups = d_model, padding = (31 - 1) // 2, stride = 1),  # Depthwise Conv1d (d_model - kernel_size + 2*padding)/stride + 1 //(512 - 35 + 2*1)//3 + 1 = 160
        nn.BatchNorm1d(d_model),  # BatchNorm
        Swish(),  # Swish activation
        nn.Conv1d(d_model, d_model, kernel_size = 1),
        nn.BatchNorm1d(d_model),  # BatchNorm
        nn.Dropout(dropout),  # Dropout
    )
    

    
    
    self.SAP = self_Attentive_pooling(d_model)
    self.pred_layer = AMSoftmax(d_model,n_spks)
    #   Change Transformer to Conformer.
    #   https://arxiv.org/abs/2005.08100
    
    # self.encoder_layer = nn.TransformerEncoderLayer(
    #   d_model = d_model, dim_feedforward = 256, nhead = 32, batch_first = True
    # )
    # self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers = 16)

    # Project the the dimension of features from d_model into speaker nums.

  def forward(self, mels):
    """
    args:
      mels: (batch size, length, 40)
    return:
      out: (batch size, n_spks)
    """
    # out: (batch size, input_channels,length, d_model)
    out = self.prenet(mels)
    # out = self.prenet(mels.unsqueeze(1))

    # out = out.permute(0, 2, 1, 3)

    # out = out.contiguous().view(out.size(0), out.size(1), out.size(2) * out.size(3))

    # out = self.project(out)
    

    for i in range(1):
        out = out + self.FFM(out)*0.5
        out = self.LN(out)
        
        out = out + self.self_attention(out)
        out = self.LN(out)

        out = out.transpose(1, 2)
        out = out + self.Conv(out)
        out = out.transpose(1, 2)
        out = self.LN(out)
        
        # out = out + self.FFM(out)*0.5
        # out = self.LN(out)
    
    # The encoder layer expect features in the shape of (length, batch size, d_model).
    # out = self.encoder(out)

    # mean pooling # 将一个“sequence”里的所有所有“word”合并成一个”word“，再丢进入FC训练
    stats = self.SAP(out)

    # out: (batch, n_spks)
    out = self.pred_layer(stats)
    return out

# Learning rate schedule
- For transformer architecture, the design of learning rate schedule is different from that of CNN.
- Previous works show that the warmup of learning rate is useful for training models with transformer architectures.
- The warmup schedule
  - Set learning rate to 0 in the beginning.
  - The learning rate increases linearly from 0 to initial learning rate during warmup period.

In [35]:
import math

import torch
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR


def get_cosine_schedule_with_warmup(
  optimizer: Optimizer,
  num_warmup_steps: int,
  num_training_steps: int,
  num_cycles: float = 0.5,
  last_epoch: int = -1,
):
  """
  Create a schedule with a learning rate that decreases following the values of the cosine function between the
  initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
  initial lr set in the optimizer.

  Args:
    optimizer (:class:`~torch.optim.Optimizer`):
      The optimizer for which to schedule the learning rate.
    num_warmup_steps (:obj:`int`):
      The number of steps for the warmup phase.
    num_training_steps (:obj:`int`):
      The total number of training steps.
    num_cycles (:obj:`float`, `optional`, defaults to 0.5):
      The number of waves in the cosine schedule (the defaults is to just decrease from the max value to 0
      following a half-cosine).
    last_epoch (:obj:`int`, `optional`, defaults to -1):
      The index of the last epoch when resuming training.

  Return:
    :obj:`torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
  """

  def lr_lambda(current_step):
    # Warmup
    if current_step < num_warmup_steps:
      return float(current_step) / float(max(1, num_warmup_steps))
    # decadence
    progress = float(current_step - num_warmup_steps) / float(
      max(1, num_training_steps - num_warmup_steps)
    )
    return max(
      0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
    )

  return LambdaLR(optimizer, lr_lambda, last_epoch)


# Model Function
- Model forward function.

In [36]:
import torch


def model_fn(batch, model, criterion, device):
  """Forward a batch through the model."""

  mels, labels = batch
  mels = mels.to(device)
  labels = labels.to(device)

  outs = model(mels)

  loss = criterion(outs, labels)

  # Get the speaker id with highest probability.
  preds = outs.argmax(1)
  # Compute accuracy.
  accuracy = torch.mean((preds == labels).float())

  return loss, accuracy


# Validate
- Calculate accuracy of the validation set.

In [37]:
from tqdm import tqdm
import torch


def valid(dataloader, model, criterion, device): 
  """Validate on validation set."""

  model.eval()
  running_loss = 0.0
  running_accuracy = 0.0
  pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit=" uttr")

  for i, batch in enumerate(dataloader):
    with torch.no_grad():
      loss, accuracy = model_fn(batch, model, criterion, device)
      running_loss += loss.item()
      running_accuracy += accuracy.item()

    pbar.update(dataloader.batch_size)
    pbar.set_postfix(
      loss=f"{running_loss / (i+1):.2f}",
      accuracy=f"{running_accuracy / (i+1):.2f}",
    )

  pbar.close()
  model.train()

  return running_accuracy / len(dataloader)


# Main function

In [38]:
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split

def parse_args():
  """arguments"""
  config = {
    "data_dir": "Dataset",
    "save_path": "./model.ckpt",
    "batch_size": 64,
    "n_workers": 0,
    "valid_steps": 2000,
    "warmup_steps": 1000,
    "save_steps": 10000,
    "total_steps": 30000,
  }

  return config


def main(
  data_dir,
  save_path,
  batch_size,
  n_workers,
  valid_steps,
  warmup_steps,
  total_steps,
  save_steps,
):
  """Main function."""
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"[Info]: Use {device} now!")

  train_loader, valid_loader, speaker_num = get_dataloader(data_dir, batch_size, n_workers)
  train_iterator = iter(train_loader)
  print(f"[Info]: Finish loading data!",flush = True)

  model = Classifier(n_spks=speaker_num).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = AdamW(model.parameters(), lr=1e-3)
  scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
  print(f"[Info]: Finish creating model!",flush = True)

  best_accuracy = -1.0
  best_state_dict = None

  pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

  for step in range(total_steps):
    # Get data
    try:
      batch = next(train_iterator)
    except StopIteration:
      train_iterator = iter(train_loader)
      batch = next(train_iterator)

    loss, accuracy = model_fn(batch, model, criterion, device)
    batch_loss = loss.item()
    batch_accuracy = accuracy.item()

    # Updata model
    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    
    # Log
    if (step + 1) % 2000 == 0:
        pbar.update()
        pbar.set_postfix(
          loss=f"{batch_loss:.2f}",
          accuracy=f"{batch_accuracy:.2f}",
          step=step + 2000,
        )
    # Do validation
    if (step + 1) % valid_steps == 0:
      pbar.close()

      valid_accuracy = valid(valid_loader, model, criterion, device)

      # keep the best model
      if valid_accuracy > best_accuracy:
        best_accuracy = valid_accuracy
        best_state_dict = model.state_dict()

      pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

    # Save the best model so far.
    if (step + 1) % save_steps == 0 and best_state_dict is not None:
        
      torch.save(best_state_dict, save_path)
      pbar.write(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")

  pbar.close()


if __name__ == "__main__":
  main(**parse_args())


[Info]: Use cuda now!
[Info]: Finish loading data!
[Info]: Finish creating model!


Train:   0% 1/2000 [00:57<32:00:49, 57.65s/ step, accuracy=0.58, loss=2.15, step=3999]
Valid:  99% 5632/5667 [00:02<00:00, 2450.67 uttr/s, accuracy=0.49, loss=2.44]
Train:   0% 1/2000 [00:57<31:44:21, 57.16s/ step, accuracy=0.59, loss=1.67, step=5999]
Valid:  99% 5632/5667 [00:02<00:00, 2437.23 uttr/s, accuracy=0.62, loss=1.83]
Train:   0% 1/2000 [00:55<30:57:33, 55.75s/ step, accuracy=0.66, loss=1.70, step=7999]
Valid:  99% 5632/5667 [00:02<00:00, 2645.18 uttr/s, accuracy=0.69, loss=1.56]
Train:   0% 1/2000 [00:52<29:17:24, 52.75s/ step, accuracy=0.81, loss=1.01, step=9999]
Valid:  99% 5632/5667 [00:02<00:00, 2678.25 uttr/s, accuracy=0.73, loss=1.42]
Train:   0% 1/2000 [00:51<28:35:50, 51.50s/ step, accuracy=0.81, loss=1.03, step=11999]
Valid:  99% 5632/5667 [00:02<00:00, 2561.49 uttr/s, accuracy=0.75, loss=1.31]
Train:   0% 0/2000 [00:00<?, ? step/s]

Step 10000, best model saved. (accuracy=0.7457)


Train:   0% 1/2000 [01:01<34:17:03, 61.74s/ step, accuracy=0.81, loss=0.98, step=13999]
Valid:  99% 5632/5667 [00:02<00:00, 2465.89 uttr/s, accuracy=0.77, loss=1.22]
Train:   0% 1/2000 [01:00<33:28:03, 60.27s/ step, accuracy=0.73, loss=1.29, step=15999]
Valid:  99% 5632/5667 [00:02<00:00, 2410.10 uttr/s, accuracy=0.78, loss=1.18]
Train:   0% 1/2000 [00:56<31:22:51, 56.51s/ step, accuracy=0.84, loss=1.05, step=17999]
Valid:  99% 5632/5667 [00:02<00:00, 2528.64 uttr/s, accuracy=0.80, loss=1.12]
Train:   0% 1/2000 [00:54<30:11:38, 54.38s/ step, accuracy=0.86, loss=0.75, step=2e+4]
Valid:  99% 5632/5667 [00:02<00:00, 2408.62 uttr/s, accuracy=0.81, loss=1.06]
Train:   0% 1/2000 [00:54<30:03:15, 54.12s/ step, accuracy=0.86, loss=0.80, step=21999]
Valid:  99% 5632/5667 [00:02<00:00, 2505.17 uttr/s, accuracy=0.82, loss=1.00]
Train:   0% 0/2000 [00:00<?, ? step/s]

Step 20000, best model saved. (accuracy=0.8182)


Train:   0% 1/2000 [00:53<29:49:52, 53.72s/ step, accuracy=0.92, loss=0.65, step=23999]
Valid:  99% 5632/5667 [00:02<00:00, 2641.39 uttr/s, accuracy=0.82, loss=1.01]
Train:   0% 1/2000 [00:54<30:21:54, 54.68s/ step, accuracy=0.84, loss=0.81, step=25999]
Valid:  99% 5632/5667 [00:02<00:00, 2491.87 uttr/s, accuracy=0.82, loss=0.97]
Train:   0% 1/2000 [00:53<29:26:43, 53.03s/ step, accuracy=0.95, loss=0.49, step=27999]
Valid:  99% 5632/5667 [00:02<00:00, 2673.76 uttr/s, accuracy=0.83, loss=0.94]
Train:   0% 1/2000 [00:50<28:07:47, 50.66s/ step, accuracy=0.89, loss=0.78, step=3e+4]
Valid:  99% 5632/5667 [00:02<00:00, 2593.93 uttr/s, accuracy=0.83, loss=0.93]
Train:   0% 1/2000 [00:54<30:15:21, 54.49s/ step, accuracy=0.89, loss=0.81, step=31999]
Valid:  99% 5632/5667 [00:02<00:00, 1913.19 uttr/s, accuracy=0.84, loss=0.93]
Train:   0% 0/2000 [00:00<?, ? step/s]

Step 30000, best model saved. (accuracy=0.8368)


# Inference

## Dataset of inference

In [39]:
import os
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset


class InferenceDataset(Dataset):
	def __init__(self, data_dir):
		testdata_path = Path(data_dir) / "testdata.json"
		metadata = json.load(testdata_path.open())
		self.data_dir = data_dir
		self.data = metadata["utterances"]

	def __len__(self):
		return len(self.data)

	def __getitem__(self, index):
		utterance = self.data[index]
		feat_path = utterance["feature_path"]
		mel = torch.load(os.path.join(self.data_dir, feat_path))

		return feat_path, mel


def inference_collate_batch(batch):
	"""Collate a batch of data."""
	feat_paths, mels = zip(*batch)

	return feat_paths, torch.stack(mels)

## Main funcrion of Inference

In [40]:
import json
import csv
from pathlib import Path
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader

def parse_args():
	"""arguments"""
	config = {
		"data_dir": "Dataset",
		"model_path": "./model.ckpt",
		"output_path": "./output.csv",
	}

	return config


def main(
	data_dir,
	model_path,
	output_path,
):
	"""Main function."""
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	print(f"[Info]: Use {device} now!")

	mapping_path = Path(data_dir) / "mapping.json"
	mapping = json.load(mapping_path.open())

	dataset = InferenceDataset(data_dir)
	dataloader = DataLoader(
		dataset,
		batch_size=1,
		shuffle=False,
		drop_last=False,
		num_workers = 0,
		collate_fn=inference_collate_batch,
	)
	print(f"[Info]: Finish loading data!",flush = True)

	speaker_num = len(mapping["id2speaker"])
	model = Classifier(n_spks=speaker_num).to(device)
	model.load_state_dict(torch.load(model_path))
	model.eval()
	print(f"[Info]: Finish creating model!",flush = True)

	results = [["Id", "Category"]]
	for feat_paths, mels in tqdm(dataloader):
		with torch.no_grad():
			mels = mels.to(device)
			outs = model(mels)
			preds = outs.argmax(1).cpu().numpy()
			for feat_path, pred in zip(feat_paths, preds):
				results.append([feat_path, mapping["id2speaker"][str(pred)]])

	with open(output_path, 'w', newline='') as csvfile:
		writer = csv.writer(csvfile)
		writer.writerows(results)


if __name__ == "__main__":
	main(**parse_args())

[Info]: Use cuda now!
[Info]: Finish loading data!
[Info]: Finish creating model!


  0%|          | 0/8000 [00:00<?, ?it/s]